In [1]:
import os
import sys 
from pathlib import Path
from dotenv import load_dotenv
import numpy as np
load_dotenv(override=True)
ROOT = Path.cwd().parent.parent.parent.resolve()

print(f"ROOT: {ROOT}")
sys.path.append(str(ROOT))

ROOT: /users/oshan/Dev/financial-document-based-agent-system


In [5]:
from QuestionAnswering.main import ExTrRAGQA
from cgcore.vectordb.milvus import MilvusDB
from cgcore.embedder.openai import OpenAIEmbedder
from cgcore.llm.openai import OpenAILlm

from cgcore.configs.vectordb.milvus import MilvusConfig
from cgcore.configs.embedder.openai import OpenAIEmbedderConfig
from cgcore.configs.llm.openai import OpenAILlmConfig

ImportError: sklearn not installed , Please install scikit-learn


In [6]:
llm_config = OpenAILlmConfig(api_key=os.getenv('OPENAI_API_KEY'))
embedder_config = OpenAIEmbedderConfig(api_key=os.getenv('OPENAI_API_KEY'), model='text-davinci-003', dimesion=os.getenv('MONGO_DB_DIMENSION'))
vectordb_config = MilvusConfig(
                collection_name=os.getenv('MILVUS_COLLECTION_NAME'),
                dimensions=1536,  # Set explicit dimension value
                )


In [7]:
openai = OpenAILlm(llm_config)
embeder = OpenAIEmbedder(embedder_config)
vectordb = MilvusDB(vectordb_config)

MilvusClient connected.
pymilvus ORM connected to localhost:19530 for setup.
Collection 'financial_documents_test' already exists. Skipping creation.


/tmp/ipykernel_1028554/1873515904.py:1: UserWarning: Parameters {'top_p'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  openai = OpenAILlm(llm_config)
/users/oshan/Dev/financial-document-based-agent-system/.venv/lib/python3.12/site-packages/langchain_openai/embeddings/base.py:313: UserWarning: WARNING! encoding_format is not default parameter.
                    encoding_format was transferred to model_kwargs.
                    Please confirm that encoding_format is what you intended.
  warnings.warn(


In [8]:
extr_rag = ExTrRAGQA(
    llm=openai,
    embedder=embeder,
    db=vectordb,
    memory="none",
    history=True
)

/users/oshan/Dev/financial-document-based-agent-system/QuestionAnswering/main.py:58: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  self.context = ConversationBufferWindowMemory(k=5)


In [9]:
extr_rag.chat("What is this document about?")

ic| f"Current Context: {current_context}": 'Current Context: '
/users/oshan/Dev/financial-document-based-agent-system/.venv/lib/python3.12/site-packages/langchain_openai/chat_models/base.py:2067: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(
ic| ruling: Rule(decision=False, related_context='', extra_questions=[])


🔍 Decision: Retrieving external documents...
[DEBUG FULL CONTENT] ID: 462951351664456937
[DEBUG FULL CONTENT] Content: '## Annual financial report and financial statements

Year to December 31, 2020

picture-1.png

## World Intellectual Property Organization

Annual Financial Report and Financial Statements

Year to December 31, 2020

## CONTENTS'
[DEBUG FULL CONTENT] Content length: 228
[DEBUG FULL CONTENT] Metadata: {'source': '/users/oshan/Dev/financial-document-based-agent-system/wipo_pub_rn2021_18e.pdf', 'processed_path': '/users/oshan/Dev/financial-document-based-agent-system/wipo_pub_rn2021_18e.md', 'parser': 'dockling_remote', 'questions': ['What is the title of the document?', 'What is the year covered in the financial report?', 'What organization is the financial report from?'], 'original_chunk_id': '3e9e1ab854df966a02423f205725d458010fa5ae5ac93d18c07ec0f0c714b450_0'}
[DEBUG] Formatted chunk: content=## Annual financial report and financial statement..., distance=0.3519101440

ic| updated_docs: [{'_id': '462951351664456937',
                    'content': '## Annual financial report and financial statements
                  '
                               '
                  '
                               'Year to December 31, 2020
                  '
                               '
                  '
                               'picture-1.png
                  '
                               '
                  '
                               '## World Intellectual Property Organization
                  '
                               '
                  '
                               'Annual Financial Report and Financial Statements
                  '
                               '
                  '
                               'Year to December 31, 2020
                  '
                               '
                  '
                               '## CONTENTS',
                    'distance': 0.35191014409065247,
           

"This document is the Annual Financial Report and Financial Statements of the World Intellectual Property Organization for the year ending December 31, 2020. It includes information on business solutions for IP offices, cash flow, internal control, auditor's report, financial statements, financial objectives and strategies, risk management, COVID-19 pandemic impact, financial performance, financial position, and expenses."

In [12]:
extr_rag.chat("What is the annual revenue for this year?")

ic| f"Current Context: {current_context}": 'Current Context: '


ic| ruling: Rule(decision=False, related_context='', extra_questions=[])


🔍 Decision: Retrieving external documents...
[DEBUG FULL CONTENT] ID: 462951351664457258
[DEBUG FULL CONTENT] Content: '|       | TOTAL REVENUE                                                                                                                             | 18,141                                                                              | 359,649                                                                             | 77,764                                                                              | 7,282                                                                               | 202                                                                                 | 5,234                                                                               | 468,272                                                                             | 468,272                                                                             |'
[DEBUG FULL CONTENT] Content length: 837


ic| updated_docs: [{'_id': '462951351664457258',
                    'content': '|       | TOTAL '
                               'REVENUE                                                                                                                             '
                               '| '
                               '18,141                                                                              '
                               '| '
                               '359,649                                                                             '
                               '| '
                               '77,764                                                                              '
                               '| '
                               '7,282                                                                               '
                               '| '
                               '202                                                

'The annual revenue for this year is 468,272 million Swiss francs.'